In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier # vecinos más cercanos para clasificación
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression



from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS

from lcode import CollinearityDropper
import time
import joblib
from sklearn import set_config
from lcode import MCFeatureEngineer, get_bmi, get_AST_ALT_Ratio, get_eyesight_mean, get_TG_HDL_Ratio, get_MAP, get_Non_HDL_Cholesterol, get_Pulse_Pressure, get_WHtR
from sklearn.ensemble import VotingClassifier

set_config(transform_output="pandas")

# Canalización final  

Notará que no hay tratamiento de outliers por ejemplo. Podria haber mejorado, potencialmente si, no lo he probado. Sera cuestion de agregar transformaciones.

## Pipeline de KNN

In [6]:
ct_knn = ColumnTransformer(
    transformers=[
        ("knn_scaler", MinMaxScaler(), ['age', 'hemoglobin', 'Gtp'])
    ],
    remainder='drop'  # Por defecto
)

pl_knn = Pipeline(steps=[
    ("scaler", ct_knn),
    ("clf_knn", KNeighborsClassifier())],
    memory=joblib.Memory(location=None)   
)


## Pipeline de Random Forest

In [7]:
pulse_pl = Pipeline(steps=[
    ("pulse_pressure",  MCFeatureEngineer(new_feature_name="Pulse_Pressure", drop_features=False,  operation=get_Pulse_Pressure)),
    ("MAP",  MCFeatureEngineer(new_feature_name="MAP", drop_features=False,  operation=get_MAP)),
])

ct1 = ColumnTransformer(
        transformers=[
            ("BMI", MCFeatureEngineer(new_feature_name="BMI", drop_features=True,  operation=get_bmi), ['height(cm)', 'weight(kg)']),
            ("WHtR", MCFeatureEngineer(new_feature_name="WHtR", drop_features=True,  operation=get_WHtR), ['Cholesterol', 'HDL']),
            ("TG_HDL_Ratio", MCFeatureEngineer(new_feature_name="TG_HDL_Ratio", drop_features=True,  operation=get_TG_HDL_Ratio), ['triglyceride', 'HDL']),
            ("Non_HDL_Cholesterol", MCFeatureEngineer(new_feature_name="Non_HDL_Cholesterol", drop_features=True,  operation=get_Non_HDL_Cholesterol), ['Cholesterol', 'HDL']),
            #("Pulse_Pressure", MCFeatureEngineer(new_feature_name="Pulse_Pressure", drop_features=False,  operation=get_Pulse_Pressure), ['systolic','relaxation']),
            # Entiendo que es mas prolijo y natural poner la linea de trabajo en un pipeline, graficar para mayor entendimiento
            ("Pulse_PL", pulse_pl, ['systolic','relaxation']),
            ("AST_ALT_Ratio", MCFeatureEngineer(new_feature_name="AST_ALT_Ratio", drop_features=False,  operation=get_AST_ALT_Ratio), ['AST','ALT']) 
        ],
        verbose_feature_names_out=False,
        remainder="passthrough"
    )


pl_rforest = Pipeline(steps=[
    ("CT1", ct1),
    ("rf_clf", RandomForestClassifier(n_estimators=90, max_depth=6, min_samples_split=40)),
],
memory=joblib.Memory(location=None))

## Pipeline regresión logística

In [8]:
preprocessor = Pipeline(steps=[
    ("feature_engineering", ct1),
    ("collinearity_drop", CollinearityDropper(min_coef=0.7, method="pearson")),
    ("scaler", StandardScaler())
])

pl = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("lr_clf", LogisticRegression()),
],
memory=joblib.Memory(location=None))


pl_lreg = Pipeline(steps=[
    ("CT1", ct1),
    ("rf_clf", RandomForestClassifier(n_estimators=90, max_depth=6, min_samples_split=40)),
],
memory=joblib.Memory(location=None))

#Agrego el estimador
preprocessor = Pipeline(steps=[
    ("feature_engineering", ct1),
    ("collinearity_drop", CollinearityDropper(min_coef=0.7, method="pearson")),
    ("scaler", StandardScaler())
])

pl_log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("lr_clf", LogisticRegression(C=0.1, l1_ratio=0.7, solver='saga', max_iter=2000))
],
memory=joblib.Memory(location=None))


In [10]:
df_smokers = pd.read_csv('https://raw.githubusercontent.com/pokengineer/DataScience/main/datasets/smokers.csv')
X = df_smokers[df_smokers.columns.drop("smoking")]
y = df_smokers["smoking"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42)

voting_clf = VotingClassifier(
    estimators=[
        ('log_reg', pl_log_reg),
        ('rforest', pl_rforest),
        ('knn', pl_knn)
    ],
    voting='soft',  # 'soft' promedia las probabilidades; 'hard' usa la mayoría simple
    n_jobs=-1
)
voting_clf


z:\git\3670-UNLaM-CD\.venv\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Parámetros:
  self[section] = content


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('log_reg', ...), ('rforest', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",-1
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineering', ...), ('collinearity_drop', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must s

In [11]:

# 2. Entrenar el ensamble completo (se entrena cada pipeline automáticamente)
voting_clf.fit(X_train, y_train)
# 3. Evaluar el desempeño
score = voting_clf.score(X_test, y_test)
print(f"Precisión del VotingClassifier: {score:.4f}")

Precisión del VotingClassifier: 0.7237
